# មេរៀន 18៖ ការការពារភ្នាក់ងារ AI ជាមួយវិក្កយបត្រសំគាល់គ្រប់គ្រាន់

## សៀវភៅកំណត់ត្រាដៃ

សៀវភៅកំណត់ត្រានេះដំណើរការតាមលំហាត់បួន៖

1. **ចុះហត្ថលេខាលើវិក្កយបត្រដំបូងរបស់អ្នក** សម្រាប់ការហៅឧបករណ៍ភ្នាក់ងារ និងផ្ទៀងផ្ទាត់វា។
2. **ខូចខាតវិក្កយបត្រ** ហើយមើលការផ្ទៀងផ្ទាត់បរាជ័យ។
3. **បង្កើតខ្សែវិក្កយបត្រ​បីចំហៀង** ហើយបញ្ជាក់ភាពចំរូងនៃខ្សែ។
4. **ដាក់ជំនួយឧបករណ៍ Microsoft Agent Framework** ដើម្បីឲ្យមុខងារនៅគ្រប់ដំណាក់កាលបញ្ចេញវិក្កយបត្រសំគាល់។

គ្រប់ធាតុគ្រីបតូក្រាហ្វីត្រូវបាននាំចូលពីបណ្ណាល័យដែលថែទាំបានល្អ (`pynacl` សម្រាប់ Ed25519, `jcs` សម្រាប់ JSON canonical តាម RFC 8785, `hashlib` ពីបណ្ណាល័យស្តង់ដាររបស់ Python សម្រាប់ SHA-256)។ លទ្ធផលវិក្កយបត្រពួកវានេះគឺ Python សាមញ្ញដែលអ្នកអាចអាន និងកែប្រែបាន។

រត់កោដដែលឡើងវិញតាមលំដាប់។ មួយផ្នែកគឺខ្លី និងមានសណ្ឋានខ្លួនឯង។


## ការតម្លើង

ដំឡើងការពឹងផ្អែកទាំងពីរ។ ពីរនេះមានអាជ្ញាបណ្ណអនុញ្ញាត (Apache-2.0 / MIT)។


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## ជំនួយការជំនួយ

ជំនួយការពីរនេះគ្រប់គ្រងការអ៊ិនគូដ base64url (ដោយគ្មាន padding) និងការប្រមូល SHA-256 នៃវត្ថុខុសៗគ្នា។ ពួកវាអនុញ្ញាតឲ្យសៀវភៅកំណត់ត្រាទាំងមូលផ្តោតលើបច្ចេកទេសនៃការទទួលយកផ្ទាល់។


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: ចុះហត្ថលេខាលើបង្កាន់ដៃដំបូងរបស់អ្នក

ស្រមៃថា ភ្នាក់ងាររបស់យើងសម្រាប់ **Contoso Travel** បានស្វែងរកតំបន់ហោះហើយតាមផ្លូវពី Sydney ទៅ Los Angeles ទៅអតិថិជនម្នាក់។ យើងចង់កត់ត្រាការហៅឧបករណ៍នេះជាបង្កាន់ដៃដែលបានចុះហត្ថលេខា ដើម្បីឱ្យអ្នកត្រួតពិនិត្យនៅអនាគតអាចផ្ទៀងផ្ទាត់វា បើឡើងដោយមិនចាំបាច់ទុកចិត្តលើយើង។

### ជំហាន 1.1: បង្កើតកូនសោចុះហត្ថលេខា

នៅក្នុងការផលិត សោចុះហត្ថលេខារបស់ភ្នាក់ងារនឹងរស់នៅក្នុងម៉ូឌុលសន្តិសុខរ៉ែ​បរិក្ខារផ្នែករឹង (HSM), Azure Key Vault, ឬហាងដែលការពារដូចគ្នា។ សម្រាប់មេរៀននេះ យើងបង្កើតកូនសោក2981ែថ្មីមួយនៅក្នុងចងចាំ។


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### ជំហាន 1.2៖ សង់វត្ថុបង់ប្រាក់ចេញវិក័យប័ត្រ

វត្ថុបង់ប្រាក់មានអ្វីគ្រប់យ៉ាងដែលយើងចង់ឲ្យវិក័យប័ត្របញ្ជាក់៖ អ្នកណាសម្តែង, ឧបករណ៍អ្វី, ជាមួយអារក្សុតអ្វី, តើអ្វីត្រូវបានតបស្នង, តាមគោលនយោបាយអ្វី និងពេលណា។ យើងធ្វើ Hash លើអារក្សុត និងលទ្ធផលជំនួសការរួមបញ្ចូលពួកវាទៅក្នុងឯកសារដូច្នេះវិក័យប័ត្រមិនបង្ហាញព័ត៌មាន​រំលោភសិទ្ធិ។


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### ជំហាន 1.3៖ ហត្ថលេខា និងប្រមូលផ្តុំវិក័យប័ត្រ

ចំនួនបីជំហាន៖

1. បំលែង payload ជាទម្រង់ canonical ដោយប្រើ JCS ដូចនេះកំណត់កម្មវិធីពីរដែលបង្កើតវិក័យប័ត្រដដែលគ្នានឹងបង្កើត bytes ដដែលគ្នាទៅ។
2. ធ្វើ hash លើ bytes canonical ដោយប្រើ SHA-256។
3. ហត្ថលេខាលើ hash ដោយប្រើកូនសោឯកជន Ed25519។

ហត្ថលេខានេះត្រូវបានភ្ជាប់ទៅនឹង payload ដើមដើម្បីបង្កើតវិក័យប័ត្រចុងក្រោយ។


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### ជំហាន 1.4៖ ផ្ទៀងផ្ទាត់វិញ្ញាបនប័ត្រ

ការផ្ទៀងផ្ទាត់បញ្ចុះដំណើរការឡើងវិញ។ យើងដកហត្ថលេខាដកចេញ កំណត់សារសំខាន់ធម្មតាឡើងវិញ ហើយពិនិត្យហត្ថលេខានេះទល់នឹងកូនសោសាធារណៈនៅក្នុងវិញ្ញាបនប័ត្រ។

អ្នកត្រួតពិនិត្យមួយដែលធ្វើការផ្ទៀងផ្ទាត់នេះមិនត្រូវការអ្វីពីយើងក្រៅតែវិញ្ញាបនប័ត្រតែមួយ។ មិនចាំបាច់ហៅសេវាកម្មណាមួយទេ មិនត្រូវស្វែងរកថតស្ដុកកូនសោទេ មិនចាំបាច់ទុកចិត្តដែរ។


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

អ្នកគួរតែឃើញ `Receipt is valid: True`។ អ្នកជំនាញបានបង្កើតកំណត់ត្រាសវនាកម្មដែលបានចុះហត្ថលេខាដោយវិធីសាស្ត្រអ៊ីគ្រីបតិស៊ីភិចលើកដំបូងរបស់ខ្លួន។


## ផ្នែកទី ២៖ ការប្តូរព័ត៌មានលើបង្កាន់ដៃ

គោលបំណងទាំងមូលនៃបង្កាន់ដៃគឺពួកវាអាចបង្ហាញពីការប្តូរបម្លែងបាន។ ចាំបង្ហាញឱ្យឃើញវា។

យើងនឹងកែប្រែត្រឹមតែអក្សរមួយចំនួនប៉ុណ្ណោះលើបង្កាន់ដៃ ហើយមើលថាការបញ្ជាក់នឹងបរាជ័យ។


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### តើអ្វីដែលកើតឡើងថ្មីៗនេះ?

នៅពេលយើងបានផ្លាស់ប្តូរ `policy_id` អត្ថបទឌីជីថល canonical បានផ្លាស់ប្តូរ។ ការប្រមូល SHA-256 នៃអត្ថបទនោះបានផ្លាស់ប្តូរ។ ហត្ថលេខា (ដែលបានធ្វើលើការប្រមូលដើម) មិនទាន់ត្រូវគ្នានឹងការប្រមូលថ្មីទេ។ ការបញ្ជាក់ត្រឹមត្រូវបង្រ្កាបចេញ `False`៕

គ្មានវិធីណាមួយដើម្បីកែប្រែវាលណាមួយនៃប័ណ្ណទទួលបានទៀងទាត់ ហើយនៅតែអាចផ្ទៀងផ្ទាត់បានទេ លុះត្រាតែអ្នកវាយប្រហារមានកូនសោឯកជន។ មុនពេលកូនសោឯកជនមាននៅក្នុងឃ្លាំងកូនសោ និងកូនសាធារណៈបានបោះពុម្ពផ្សាយ ការបំភាន់ស្លៀកបំពាក់គឺមិនអាចលាក់បានទេ។

សូមសាកល្បងដោយខ្លួនអ្នក៖ កែប្រែ `tool_name` ឬ `agent_id` ឬ `timestamp` នៅក្នុងក្រុមខាងលើ ហើយរត់ម្តងទៀត។ ការផ្លាស់ប្តូរមួយៗបង្កើតប័ណ្ណទទួលមិនត្រឹមត្រូវទេ។


## ផ្នែកទី 3៖ ចងក្រដាសបង់ប្រាក់ជាសន្ទស្សន៍ជាប់គ្នា

ក្រដាសបង់ប្រាក់មួយគត់ការពារការប្រតិបត្តិមួយ។ ច្រើនភ្នាក់ងារធ្វើការប្រតិបត្តិជាច្រើន។ ដើម្បីឱ្យលំដាប់ទាំងមូលបង្ហាញសញ្ញា​បម្លែង យើងភ្ជាប់ក្រដាសបង់ប្រាក់នីមួយៗទៅក្រដាសបង់ប្រាក់មុន ដោយបញ្ចូល hash របស់ក្រដាសបង់ប្រាក់មុននៅក្នុងទិន្នន័យរបស់ក្រដាសបង់ប្រាក់ថ្មី។

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

បើមាននរណាមួយដកយកឬប្តូរលំដាប់ក្រដាសបង់ប្រាក់ មេជួរនឹងដាច់នៅចំណុចនោះពិតប្រាកដ។ ការផ្ទៀងផ្ទាត់ក្រដាសបង់ប្រាក់បន្ទាប់​បាញ់បរាជ័យ ព្រោះ `previous_receipt_hash` មិនត្រូវគ្នាជាមួយ hash ដែលពិតរបស់បុព្វជនទេ។


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

ឥឡូវនេះផ្អាកខ្សែសង្វាក់ដោយកែចេញវិញ្ញាបនបត្រខាងកណ្តាលហើយពិនិត្យម្តងទៀតវិញ។ វិញ្ញាបនបត្រដែលបានកែប្រែក្លាយជាប់បរាជ័យនៅក្នុងការត្រួតពិនិត្យហត្ថលេខារបស់វា ហើយវិញ្ញាបនបត្របន្ទាប់មួយក៏បរាជ័យក្នុងការត្រួតពិនិត្យតំណខ្សែសង្វាក់របស់វា (ដោយសារតែលេខឌីហាសរបស់`previous_receipt_hash`មិនត្រូវគ្នាហើយជាមួយលេខឌីហាសដែលបានកែប្រែរបស់វិញ្ញាបនបត្រខាងកណ្តាល)។


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

វិក្កយបត្រ 0 ស្ថិតនៅក្នុងស្ថានភាពផ្ទៀងផ្ទាត់បាន (វាមិនត្រូវបានកែប្រែ និងមិនមានវិក្កយបត្រមុនទៀតសម្រាប់ពឹងផ្អែក)។ វិក្កយបត្រ 1 បរាជ័យក្នុងការត្រួតពិនិត្យហត្ថលេខារបស់វា ព្រោះយើងបានប្ដូរ `tool_args_hash`។ វិក្កយបត្រ 2 បរាជ័យក្នុងការត្រួតពិនិត្យខ្សែចង្វាក់ ព្រោះ `previous_receipt_hash` របស់វាត្រូវបានគណនាប្រកួតប្រជែងនឹងវិក្កយបត្រដើម (ដែលឥឡូវនេះត្រូវបានកែប្រែ) 1។

ទោះបីជាអ្នកគំរាមកំហែងបានធ្វើហត្ថលេខាឡើងវិញលើវិក្កយបត្របានកែប្រែ 1 (ដែលពួកគេមិនអាចធ្វើបានដោយគ្មានកូនសោឯកជន) ការបំភ្លេចខ្សែចង្វាក់នៅក្នុងវិក្កយបត្រ 2 នឹងត្រូវបង្ហាញការបញ្ឆោតបញ្ឆណ៍។ ដើម្បីលាក់ការផ្លាស់ប្តូរ អ្នកគំរាមកំហែងនឹងត្រូវធ្វើហត្ថលេខាឡើងវិញលើវិក្កយបត្ររាល់អ្វីៗចាប់ពីចំណុចកែប្រែទៅមុខ ដែលតម្រូវឱ្យមានការកាន់កាប់កូនសោឯកជន។


## ផ្នែកទី 4៖ បង្រួមការហៅឧបករណ៍ភ្នាក់ងារជាមួយការចុះហត្ថលេខារបបខ្ទង់

ក្នុងការតម្លើងជាក់ស្តែង អ្នកមិនចង់ឲ្យអ្នកនិពន្ធភ្នាក់ងារពីរបៀបរៀងរាល់ទេមកហៅ `make_receipt` ទេ។ អ្នកចង់ឲ្យការចុះហត្ថលេខារបបខ្ទង់ជាដំណើរការដែលធ្វើដោយស្វ័យប្រវត្តិសម្រាប់ការហៅឧបករណ៍រាល់មួយ។

នេះជាគំរូងាយស្រួលបំផុត៖ ថ្នាក់បង្រួមដែលយកមុខងារឧបករណ៍ណាមួយដែលអាចហៅបាន ហើយបង្វិលវាទៅជាបំណែកដែលបញ្ចេញបបុកខ្ទង់។ វាអាចបត់បែនទៅនឹងស៊ុមភ្នាក់ងារណាមួយ ហូទ្រកសាំង Microsoft Agent Framework (`agent_framework.azure`) ផងដែរ។

បើអ្នកមិនមានគម្រោង Azure AI Foundry ដែលបានកំណត់ឡើងទេ ការតម្រៀបកន្លែងក្នុងស្រុកខាងក្រោមនេះនៅតែបង្ហាញគំរូវា។


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### ផ្គូព្យជាមួយស៊ុម Microsoft Agent Framework

បំពង់ `ReceiptedTool` នៅលើគឺមិនបន្តឹងទៅស៊ុមណាមួយទេ។ ដើម្បីប្រើវាក្នុងភ្នាក់ងារដែលបានសម្រួលជាមួយ Microsoft Agent Framework, ចោលការចុះបញ្ជីមុខងារដែលបានបំពង់ជាឧបករណ៍។ រូបរាងកខ្វក់ (អ្នកនឹងជំនួស mock ជាការចុះបញ្ជីឧបករណ៍ Azure AI Foundry ពិតប្រាកដ):

```python
# កូដស្មើដែលបង្ហាញពីរាងសម្ពែងការចងក្រង។
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="អ្នក​ជា​អ្នកតំណាងធ្វើដំណើរ Contoso ...",
#     tools=[receipted_lookup],   # ឧបករណ៍ដែលបានបញ្ជូលជាពីរមុខ មិនមែនមុខងារចម្បងទេ
# )
# response = agent.run("ស្វែងរកជើងហោះហើរពី Sydney ទៅ Los Angeles ក្នុងខែមិថុនា។")
#
# # បន្ទាប់ពីរត់ហើយ គ្រប់ការហៅឧបករណ៍ដែលភ្នាក់ងារបានធ្វើ មានវិក្កយបត្រចុះហត្ថលេខា:
# audit_chain = receipted_lookup.receipts
```
  
ស៊ុមភ្នាក់ងារមិនចាំបាច់ដឹងអ្វីអំពីបង្កាន់ដៃទេ។ ការចុះហត្ថលេខាបង្កាន់ដៃត្រូវបានបំពង់ជុំវិញឧបករណ៍ មិនមែនភ្ជាប់នៅក្នុងស៊ុមទេ។ នេះជាវិធីដែលអ្នកបន្ថែមប្រភពដើមទៅកូដភ្នាក់ងារដដែលដោយគ្មានការសរសេរឡើងវិញភ្នាក់ងារ។


## សង្ខេប និងបញ្ហាបណ្ដុះបណ្ដាល

អ្នកបាន:

- បង្កើតគូកូនសោ Ed25519 មួយ។
- សាងសង់ និងចុះហត្ថលេខាលើបង្កាន់ដៃសម្រាប់ការហៅឧបករណ៍ភ្នាក់ងារ។
- វាយតម្លៃបង្កាន់ដៃក្រៅបណ្តាញដោយប្រើសោសាធារណៈតែប៉ុណ្ណោះ។
- បំភ្លេចបង្កាន់ដៃមួយ ហើយបានមើលឃើញការផ្ទៀងផ្ទាត់បរាជ័យ។
- សាងសង់ស្សេរីរបស់បង្កាន់ដៃបីដោយបានភ្ជាប់ជាក្រចងហាស៍។
- បំភ្លេចនៅកណ្តាលក្រចង ហើយបានមើលឃើញទាំងការបរាជ័យនៃហត្ថលេខា និងការបរាជ័យក្នុងភ្ជាប់ក្រចង។
- ខ្ចុះឧបករណ៍មួយជាមួយការចុះហត្ថលេខាផ្តល់បង្កាន់ដៃដោយស្វ័យប្រវត្តិ។

**បញ្ហាបណ្ដុះបណ្ដាលបន្ថែម។** ពង្រីកគំរូបង្កាន់ដៃជាមួយវាល `request_id` (UUID សម្រាប់កំណត់តាមការចែកចាយ)។ បន្ទាន់សម័យ `make_receipt` ដើម្បីរួមបញ្ចូលវា ហើយបញ្ជាក់ថាបង្កាន់ដៃនៅតែអាចផ្ទៀងផ្ទាត់ពីដើមចុង។ បន្ទាប់មកកែប្រែវាលនេះបន្ទាប់ពីចុះហត្ថលេខា និងបញ្ជាក់ថាការផ្ទៀងផ្ទាត់បរាជ័យ។ វា បង្ខំឱ្យអ្នកយល់ពីរបៀបដែលបៃមួយៗនៃកូដបញ្ចូលប្រព័ន្ធសរសេរដំណើរការអ្នកចូលរួមក្នុងហត្ថលេខា។ 

**ព្រំប្រទល់សំខាន់។** បង្កាន់ដៃបញ្ជាក់ពីរឿងបី និងតែបី៖ ការចាត់ទុក (កូនសោនេះបានចុះហត្ថលេខាលើមាតិការនេះ), សុវត្ថិភាព (មាតិកានេះមិនបានផ្លាស់ប្ដូរពីពេលចុះហត្ថលេខា), និងលំដាប់ (បង្កាន់ដៃនេះបានមកបន្ទាប់ពីបង្កាន់ដៃផ្សេង)។ វាមិនបញ្ជាក់ថាទំនាក់ទំនងឬសកម្មភាពរបស់ភ្នាក់ងារជាដូចម្តេចទេ, ឬថានយោបាយដែលបានរៀបចំក្នុង `policy_id` ត្រូវបានវាយតម្លៃពិតប្រាកដ, ឬថាភ្នាក់ងារបានអនុវត្តតាមច្បាប់គ្រប់យ៉ាង។ បង្កាន់ដៃគឺជាគ្រឹះ។ ពាណិជ្ជកម្មគឺជាប្រព័ន្ធដែលអ្នកបង្កើតនៅលើវា។

អាន README មេរៀនម្តងទៀតជាមួយព្រំប្រទល់នេះម្តង។ កំហុសធម្មតា និងទូទៅបំផុតដែលក្រុមធ្វើនោះគឺគិតថា "យើងមានបង្កាន់ដៃ" មានន័យថា "យើងមានការគ្រប់គ្រង។" វាមិនមែនទេ។ បង្កាន់ដៃធ្វើឲ្យសកម្មភាពភ្នាក់ងារអាចត្រូវបានត្រួតពិនិត្យបាន។ វាមិនធ្វើឲ្យវាត្រឹមត្រូវឡើយ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
